# Mô hình dựa trên mệnh đề + SkinCAP — chạy huấn luyện trên Google Colab

Notebook này chạy `main_proposition.py` (nhánh `RAD-colab-enhanced`) với **đúng cấu hình đã dùng cho RAD gốc** (lần chạy `20261005_091021`): batch 4, tích lũy gradient 4, 20 epoch, warmup 10. Nhờ vậy kết quả so sánh được trực tiếp với RAD.

1. Clone đúng nhánh `RAD-colab-enhanced` và cài dependency.
2. Kiểm tra GPU, Google Drive, ClinicalBERT và dữ liệu SkinCAP.
3. Xác thực một batch dữ liệu thật và tập mệnh đề.
4. Tạo thư mục chạy và cấu hình (giống RAD).
5. Khởi chạy hoặc tiếp tục đúng thư mục chạy đó.
6. Kiểm tra checkpoint và metrics đã lưu trên Drive.
7. So sánh metrics với RAD gốc.

**Trước khi chạy:** chọn **Runtime → Change runtime type → GPU**.

> Mỗi thí nghiệm mới nằm trong `Proposition-outputs/P1-20ep/YYYYMMDD_HHMMSS`. Muốn resume, dán `RUN_ID` đã in ra vào ô 3.


## 1. Chuẩn bị Python và repository

Clone **nhánh `RAD-colab-enhanced`** (nhánh `main` chỉ có RAD gốc, không có code mệnh đề). Nếu repository đã có trong phiên Colab hiện tại, ô này chuyển sang đúng nhánh và kéo commit mới nhất.


In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version)

REPO_DIR = Path("/content/RAD-colab")
REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
BRANCH = "RAD-colab-enhanced"

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}

!git -C {REPO_DIR} log --oneline -3
assert (REPO_DIR / "main_proposition.py").is_file(), "Sai nhánh: không thấy main_proposition.py"
print(f"✅ Repository: {REPO_DIR} (nhánh {BRANCH})")


## 2. Cài dependency và kiểm tra GPU

`requirements-colab-skin.txt` không cài lại `torch`/`torchvision`, để giữ cặp CUDA tương thích mà Colab đang cung cấp.

In [ ]:
%cd /content/RAD-colab
%pip install -q -r requirements-colab-skin.txt

import importlib
from importlib.metadata import version

modules = {
    "yaml": "PyYAML",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "pydicom": "pydicom",
    "nibabel": "nibabel",
    "tensorboardX": "tensorboardX",
    "transformers": "transformers",
    "timm": "timm",
    "einops": "einops",
    "peft": "peft",
    "matplotlib": "matplotlib",
}

for module, package in modules.items():
    importlib.import_module(module)
    print(f"✅ {module:15} {version(package)}")

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount Drive, tải ClinicalBERT khi cần và chọn lần chạy

Chỉ sửa các đường dẫn riêng nếu vị trí dữ liệu hoặc output trên Drive thay đổi. Nếu `/content/models/Bio_ClinicalBERT` chưa tồn tại trong runtime Colab hiện tại, notebook sẽ tải model gốc `emilyalsentzer/Bio_ClinicalBERT` từ Hugging Face vào đó. Runtime Colab là tạm thời nên việc tải lại sau khi reset runtime là bình thường.

- Để `RUN_ID = None` khi chủ động bắt đầu một thí nghiệm mới. Notebook tạo và in một timestamp dạng `YYYYMMDD_HHMMSS`.
- Khi Colab ngắt hoặc reset runtime, dán timestamp đã in vào `RUN_ID` để tiếp tục **đúng** lần chạy đó.
- Không để `RUN_ID = None` khi muốn resume, vì điều này sẽ tạo thí nghiệm mới.

In [ ]:
from datetime import datetime, timezone
from google.colab import drive
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

SKINCAP_ROOT = Path("/content/drive/MyDrive/Cao học/Experiments/SkinCAP")
TRAIN_CSV = SKINCAP_ROOT / "skincap_47_train.csv"
VAL_CSV = SKINCAP_ROOT / "skincap_47_val.csv"
TEST_CSV = SKINCAP_ROOT / "skincap_47_test.csv"
IMAGE_ROOT = SKINCAP_ROOT / "skincap"

# Colab storage is temporary. Download the original ClinicalBERT model only when absent.
BERT_MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
BERT_DIR = Path("/content/models/Bio_ClinicalBERT")
if not (BERT_DIR / "config.json").is_file():
    from huggingface_hub import snapshot_download

    BERT_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"Đang tải {BERT_MODEL_ID} vào {BERT_DIR}...")
    snapshot_download(
        repo_id=BERT_MODEL_ID,
        local_dir=str(BERT_DIR),
        local_dir_use_symlinks=False,
    )
else:
    print(f"✅ Đã có ClinicalBERT: {BERT_DIR}")

EXPERIMENT_NAME = "P1-20ep"
RUNS_ROOT = Path("/content/drive/MyDrive/Cao học/Proposition-outputs") / EXPERIMENT_NAME
RUN_ID = None   # Thí nghiệm mới. Resume: dán RUN_ID đã in ra, ví dụ "20261007_101500".

for label, path, check in [
    ("Train CSV", TRAIN_CSV, Path.is_file),
    ("Validation CSV", VAL_CSV, Path.is_file),
    ("Test CSV", TEST_CSV, Path.is_file),
    ("Thư mục ảnh", IMAGE_ROOT, Path.is_dir),
    ("ClinicalBERT", BERT_DIR, Path.is_dir),
]:
    ok = check(path)
    print(f"{'✅' if ok else '❌'} {label}: {path}")
    assert ok, f"Không tìm thấy {label}: {path}"

png_count = len(list(IMAGE_ROOT.glob("*.png")))
print("Số ảnh PNG:", png_count)
assert png_count > 0, "Không có ảnh PNG trong IMAGE_ROOT."

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=False, exist_ok=False)
    print(f"✅ Tạo lần chạy mới: {RUN_ID} (UTC)")
    print("⚠️ Lưu RUN_ID này. Dán lại vào biến RUN_ID để resume sau khi runtime reset.")
else:
    RUN_DIR = RUNS_ROOT / RUN_ID
    latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
    assert latest_checkpoint.is_file(), (
        f"Không thể resume: không tìm thấy checkpoint hoàn chỉnh {latest_checkpoint}"
    )
    print(f"✅ Resume lần chạy: {RUN_ID}")

for directory in ["checkpoints", "metrics", "predictions", "tensorboard", "logs"]:
    (RUN_DIR / directory).mkdir(exist_ok=True)

print("Run directory:", RUN_DIR)

## 4. Kiểm tra ClinicalBERT và một batch SkinCAP

Ô này không train. Nó xác nhận model BERT cục bộ có hidden size 768 và loader đọc đúng CSV, ảnh và 47 nhãn trước khi bắt đầu thí nghiệm dài.

In [ ]:
%cd /content/RAD-colab

from transformers import AutoConfig, AutoTokenizer
from torch.utils.data import DataLoader
from dataset.dataset_entity import Skin_Train_Dataset
from dataset.test_dataset import Skin_Test_Dataset

bert_config = AutoConfig.from_pretrained(BERT_DIR, local_files_only=True)
AutoTokenizer.from_pretrained(BERT_DIR, local_files_only=True)
assert bert_config.hidden_size == 768, (
    f"ClinicalBERT phải có hidden_size=768, nhận được {bert_config.hidden_size}."
)
print("✅ ClinicalBERT:", bert_config.model_type, "hidden_size=", bert_config.hidden_size)

train_dataset = Skin_Train_Dataset(TRAIN_CSV, image_res=512, image_root=IMAGE_ROOT)
val_dataset = Skin_Test_Dataset(VAL_CSV, image_res=512, image_root=IMAGE_ROOT)
test_dataset = Skin_Test_Dataset(TEST_CSV, image_res=512, image_root=IMAGE_ROOT)
batch = next(iter(DataLoader(train_dataset, batch_size=2, num_workers=0)))

print("Số mẫu train:", len(train_dataset))
print("Số mẫu val :", len(val_dataset))
print("Số mẫu test :", len(test_dataset))
print("Batch image :", tuple(batch["image"].shape))
print("Batch label :", tuple(batch["label"].shape))
assert tuple(batch["image"].shape) == (2, 3, 512, 512)
assert tuple(batch["label"].shape) == (2, 47)
print("✅ Dữ liệu SkinCAP sẵn sàng.")

from proposition.data.knowledge_base import load_proposition_kb
import pandas as pd

KB_PATH = REPO_DIR / "kb_builder/outputs/final/kb_propositions.json"
labels = list(pd.read_csv(TRAIN_CSV, nrows=0).columns[2:])
kb = load_proposition_kb(KB_PATH, labels)
print("Số bệnh:", kb.num_diseases, "| Số mệnh đề:", kb.num_propositions,
      "| Polarity −1:", int((kb.prop_polarity == -1).sum()))
assert kb.num_diseases == 47 and kb.num_propositions == 1436
print("✅ Tập mệnh đề khớp đúng thứ tự 47 nhãn trong CSV.")


## 5. Tạo cấu hình huấn luyện

Cấu hình **giống hệt lần chạy RAD gốc** `20261005_091021`: `batch_size=4`, `grad_accumulation_steps=4`, 20 epoch, warmup 10. Không đổi các số này nếu muốn so sánh công bằng với RAD.


In [ ]:
import yaml

with open("configs/skin.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config.update({
    "batch_size": 4,
    "test_batch_size": 4,
    "num_workers": 2,
    "test_num_workers": 2,
    "grad_accumulation_steps": 4,
})
config["schedular"].update({
    "epochs": 20,
    "warmup_epochs": 10,
})

FULL_CONFIG = RUN_DIR / "proposition-20ep.yaml"
FULL_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(FULL_CONFIG.read_text(encoding="utf-8"))

## 6. Khởi chạy hoặc resume an toàn

Sau mỗi **epoch hoàn chỉnh** (train + validation), code lưu trên Drive:

- `checkpoints/latest.pt` (để resume) và `checkpoints/best.pt` (epoch có macro-F1 val cao nhất);
- `metrics/metrics_history.csv`, `metrics/latest_metrics.json`;
- `predictions/val_epoch_XXX.npz`.

Khi train xong, code test `best.pt` một lần và ghi `final_metrics.json`, `predictions/test_best.npz`, `run_config.json` và thư mục `checklist/`.

Trong lúc chạy, ô này không in tiến độ từng batch; muốn xem tiến độ thì chạy ô 7 ở một tab khác hoặc sau khi một epoch hoàn thành.


In [ ]:
import os
import subprocess
import sys

latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
if latest_checkpoint.is_file():
    print("✅ Sẽ resume từ checkpoint hoàn chỉnh:", latest_checkpoint)
else:
    print("ℹ️ Chưa có checkpoint: thí nghiệm sẽ bắt đầu từ epoch 1.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.chdir(REPO_DIR)

# Chỉ truyền các tham số main_proposition.py nhận. Các hệ số còn lại (tau_att=0.1,
# PECL 0.1/0.001, nhiệt độ 0.5/2.0, lambda_label=1) đã là mặc định đúng theo tài liệu.
command = [
    sys.executable,
    "main_proposition.py",
    "--config", str(FULL_CONFIG),
    "--train_csv", str(TRAIN_CSV),
    "--val_csv", str(VAL_CSV),
    "--test_csv", str(TEST_CSV),
    "--image_root", str(IMAGE_ROOT),
    "--output_dir", str(RUN_DIR),
    "--bert_model_name", str(BERT_DIR),
    "--kb_path", str(KB_PATH),
    "--max_length", "512",
    "--embed_dim", "768",
    "--topk", "1", "2", "3",
    "--seed", "42",
    "--run_id", "P1_s42",
]

print("Lệnh sẽ chạy:")
print(" ".join(repr(part) if " " in part else part for part in command))
subprocess.run(command, check=True)


## 7. Kiểm tra trạng thái đã lưu trên Drive

Sau khi một epoch hoàn thành, chạy ô này để xem checkpoint có thể resume, metrics mới nhất và phần cuối của log. Ô này chỉ đọc file; không bắt đầu huấn luyện.

In [ ]:
import json
import pandas as pd

latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
history_path = RUN_DIR / "metrics" / "metrics_history.csv"
latest_metrics_path = RUN_DIR / "metrics" / "latest_metrics.json"
log_path = RUN_DIR / "logs" / "training.log"
best_checkpoint = RUN_DIR / "checkpoints" / "best.pt"
final_metrics_path = RUN_DIR / "final_metrics.json"

print("Run directory:", RUN_DIR)
print("Latest checkpoint:", latest_checkpoint if latest_checkpoint.is_file() else "chưa có")
print("Best checkpoint:", best_checkpoint if best_checkpoint.is_file() else "chưa có")
print("Milestone checkpoints:", [path.name for path in sorted((RUN_DIR / "checkpoints").glob("epoch_*.pt"))])

if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history.tail(5))
else:
    print("ℹ️ Chưa có metrics_history.csv; epoch đầu tiên chưa hoàn thành.")

if latest_metrics_path.is_file():
    print("\nLatest metrics:")
    print(json.dumps(json.loads(latest_metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

if log_path.is_file():
    print("\n20 dòng log cuối:")
    print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))
if final_metrics_path.is_file():
    print("\nFinal test metrics:")
    print(json.dumps(json.loads(final_metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("\nChưa có final_metrics.json; file này chỉ được tạo sau khi train xong và test best.pt một lần.")


## 8. So sánh với RAD gốc

Đọc `final_metrics.json` của lần chạy RAD (`20261005_091021`) và của lần chạy này. RAD báo ba đầu ra: `fused` (đầu ra chính thức của RAD), `guideline` và `label`. Mô hình mệnh đề báo `proposition` (đầu ra chính) và `label_aux` (chỉ tham khảo).


In [ ]:
import json
import pandas as pd

RAD_RUN_DIR = Path("/content/drive/MyDrive/Cao học/RAD-outputs/skin-full-30ep/20261005_091021")
rad = json.loads((RAD_RUN_DIR / "final_metrics.json").read_text(encoding="utf-8"))
prop = json.loads((RUN_DIR / "final_metrics.json").read_text(encoding="utf-8"))

METRICS = ["top1_accuracy", "top2_accuracy", "top3_accuracy", "mrr_at_2", "mrr_at_3",
           "macro_mrr_at_3", "macro_f1", "balanced_accuracy", "macro_auc", "mAP"]
LEGACY = ["mean_f1", "mean_precision", "mean_recall", "mean_accuracy", "subset_accuracy"]

def row(block):
    values = {name: block[name] for name in METRICS}
    values.update({f"rad_{name}": block["rad_legacy_metrics"][name] for name in LEGACY})
    return values

columns = {
    "RAD fused": row(rad["fused"]),
    "RAD guideline": row(rad["guideline"]),
    "Mệnh đề (chính)": row(prop["proposition"]),
}
if "label_aux" in prop:
    columns["Mệnh đề label_aux"] = row(prop["label_aux"])
table = pd.DataFrame(columns).round(4)
table["Δ (mệnh đề − RAD fused)"] = (table["Mệnh đề (chính)"] - table["RAD fused"]).round(4)
display(table)
print("Best epoch — RAD:", rad.get("best_epoch"), "| Mệnh đề:", prop.get("best_epoch"))
